# CourtListener Training Data Builder

Pulls public bankruptcy filings from CourtListener and turns them into training data for two stages:

- **Time entries:** real attorney billing narratives, used as grounding for synthetic data generation.
- **Fee examiner reports:** what examiners flag and why, used to define compliance rules.

**Before running:**
1. Add a CourtListener API token to a `.env` file in this folder: `COURTLISTENER_TOKEN=your_token_here`. The token is under your CourtListener profile.
2. Download `data/raw/raw_scrape.json.gz` from the shared Google Drive folder into `data/raw/`.
3. Keep `RUN_SCRAPE = False` unless new cases were added to `cases.csv`. A full scrape takes hours.

**Outputs** (all in `data/final/`):

| File | Contents |
|---|---|
| `time_entries_all.csv.gz` | Every parsed entry, with confidence and role flags. Full audit trail. `pd.read_csv` opens it directly. |
| `time_entries_clean_balanced.csv` | Clean attorney entries, capped per case, in the same column format as the synthetic dataset. Use for synthetic data generation. |
| `examiner_reports.json.gz` | Deduplicated fee examiner and fee committee reports, full text. |
| `examiner_paragraphs.csv` | Report paragraphs tagged by violation category. Starting point for compliance rules. |

**Keyword lists:** every list in this notebook is either a search term or something all bankruptcy filings share, such as court stamps, numbered paragraphs, UTBMS task codes, and the U.S. Trustee guideline categories. None are written around a specific firm or case. If a new case breaks something, look for the general pattern before adding a one-off fix.

## Setup
Imports and configuration.

In [1]:
import os
import re
import gzip
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
# Set to True only after adding new cases to cases.csv
RUN_SCRAPE = False

# Max entries per case in the balanced set, so one large case doesn't dominate
MAX_ENTRIES_PER_CASE = 3000

RAW_DIR = "data/raw"
DOC_CACHE_DIR = "data/raw/docs"
INTERMEDIATE_DIR = "data/intermediate"
FINAL_DIR = "data/final"

RAW_PATH = f"{RAW_DIR}/raw_scrape.json.gz"
WITH_TEXT_PATH = f"{INTERMEDIATE_DIR}/docs_with_text.json.gz"
SECTIONS_PATH = f"{INTERMEDIATE_DIR}/time_entry_sections.json.gz"

for folder in [RAW_DIR, DOC_CACHE_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    os.makedirs(folder, exist_ok=True)

### File Helpers
Large files are saved gzipped to stay under GitHub's 100MB file limit. These helpers handle both `.json` and `.json.gz`.

In [3]:
def load_json(path):
    """Loads a JSON file, gzipped or not."""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def save_json(data, path):
    """Saves to JSON. Gzips the file if the path ends in .gz."""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "wt", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

### Boilerplate Cleanup
Filings repeat text at the top or bottom of each page, such as court stamps, firm letterhead, and invoice headers. Instead of listing these per firm, the cleanup finds lines that repeat at the edges of most pages and strips them. RECAP text marks page breaks with `\x0c`. `COURT_STAMP` covers documents without page breaks, since every bankruptcy court uses the same stamp format.

In [4]:
COURT_STAMP = re.compile(r"^.*(Doc \d+\s+Filed|Page \d+ of \d+|Pg \d+ of \d+|Main Document).*$", re.M | re.I)


def normalize_line(line):
    """Lowercases a line and swaps digits for # so 'Page 3 of 40' and 'Page 4 of 40' match."""
    return re.sub(r"\d+", "#", " ".join(line.split())).lower()


def find_page_boilerplate(text, edge_lines=4, min_share=0.5):
    """Finds lines that repeat in the top or bottom few lines of at least half the pages."""
    pages = [p for p in text.split("\x0c") if p.strip()]
    if len(pages) < 3:
        return set()

    counts = Counter()
    for page in pages:
        lines = [l for l in page.split("\n") if l.strip()]
        counts.update({normalize_line(l) for l in lines[:edge_lines] + lines[-edge_lines:]})

    threshold = max(3, min_share * len(pages))
    return {line for line, n in counts.items() if n >= threshold}

In [5]:
def strip_boilerplate(text):
    """Removes repeated page headers, footers, and court stamps from a document."""
    boilerplate = find_page_boilerplate(text)
    if boilerplate:
        text = "\n".join(l for l in text.split("\n") if normalize_line(l) not in boilerplate)
    return COURT_STAMP.sub("", text)

## Cases and Search Settings
Cases are listed in `cases.csv` with columns `name`, `docket_number`, and `court` (CourtListener court ID). Cases with an appointed fee examiner or fee committee are the most useful.

In [6]:
cases_df = pd.read_csv("cases.csv")
CASES = cases_df.to_dict("records")
print(f"{len(CASES)} cases in cases.csv")

# Search terms for each docket
KEYWORDS = [
    "fee examiner",
    "interim compensation",
    "interim fee application",
    "final fee application",
    "monthly fee statement",
    "fee committee",
    "objection to fee",
    "compensation and reimbursement",
]

# Only these get a second API call for full text, which keeps the request count down
HIGH_VALUE_TERMS = ["fee examiner", "fee committee", "examiner's report", "monthly fee statement", "interim compensation"]

14 cases in cases.csv


In [7]:
TOKEN = os.environ.get("COURTLISTENER_TOKEN")
HEADERS = {"Authorization": f"Token {TOKEN}"}
API_ROOT = "https://www.courtlistener.com/api/rest/v4"

REQUEST_DELAY = 3.5         # Seconds between calls, stays under ~20 requests per minute
DAILY_REQUEST_LIMIT = 5000  # Safety stop per session
request_count = 0

if RUN_SCRAPE and not TOKEN:
    raise ValueError("No COURTLISTENER_TOKEN found. Add it to your .env file.")

## API Helpers
`safe_get` paces requests and retries when rate limited. `get_doc_cached` saves every fetched document to `data/raw/docs/`, so reruns never download the same document twice.

In [8]:
def safe_get(url, params=None, retries=3, backoff=2):
    """Requests a URL with pacing and retries. Returns the JSON, or None if it keeps failing."""
    global request_count

    if request_count >= DAILY_REQUEST_LIMIT:
        print("Hit the request safety stop. Raise DAILY_REQUEST_LIMIT to continue.")
        return None

    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            request_count += 1
            time.sleep(REQUEST_DELAY)

            if resp.status_code == 429:
                wait = backoff * (2 ** attempt)
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            if 400 <= resp.status_code < 500:
                # 404, 403, etc. Retrying won't help
                print(f"Got {resp.status_code} for {url}, skipping")
                return None

            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"Request failed ({e}), retry {attempt + 1}/{retries}")
            time.sleep(backoff)

    print(f"Giving up on {url}")
    return None

In [9]:
def get_doc_cached(doc_id):
    """Fetches one RECAP document, using the local cache when available."""
    path = f"{DOC_CACHE_DIR}/{doc_id}.json"
    if os.path.exists(path):
        return load_json(path)

    data = safe_get(f"{API_ROOT}/recap-documents/{doc_id}/")
    if data:
        save_json(data, path)
    return data

## Scraping Functions
Searches each docket for fee-related filings (filtered server-side), flattens the results into a list of documents, then fetches full text for high-value documents.

In [10]:
def search_docket(docket_number, court):
    """Searches one docket for fee-related filings. Returns (results, incomplete)."""
    keyword_clause = " OR ".join(f'"{kw}"' for kw in KEYWORDS)
    query = f'docketNumber:"{docket_number}" AND court_id:{court} AND ({keyword_clause})'

    url = f"{API_ROOT}/search/"
    params = {"q": query, "type": "rd", "page_size": 100}
    results, incomplete = [], False

    while url:
        data = safe_get(url, params=params)
        if not data:
            incomplete = True
            break
        results.extend(data.get("results", []))
        url = data.get("next")
        params = None  # The "next" URL already includes the params

    return results, incomplete

In [11]:
def flatten_documents(search_results):
    """Pulls documents out of search results. Attachments (like "Exhibit B") get their
    parent's description added, so it's clear what they belong to."""
    docs = []
    for r in search_results:
        nested = r.get("recap_documents")
        if not nested:
            docs.append(r)
            continue
        parent_desc = r.get("description", "") or ""
        for d in nested:
            d["description"] = f"{parent_desc} | {d.get('description', '') or ''}"
            docs.append(d)
    return docs


def is_high_value(description):
    """True if the document is worth an extra API call to get its text."""
    d = (description or "").lower()
    return any(term in d for term in HIGH_VALUE_TERMS)

In [12]:
def fetch_text(doc):
    """Returns (plain_text, absolute_url, failed) for one document. Fetches full text only when worthwhile."""
    plain_text = doc.get("plain_text")
    absolute_url = doc.get("absolute_url")
    worth_fetching = doc.get("is_available") and not plain_text and is_high_value(doc.get("description"))

    if worth_fetching and doc.get("id"):
        full = get_doc_cached(doc["id"])
        if not full:
            return plain_text, absolute_url, True
        plain_text = full.get("plain_text")
        absolute_url = full.get("absolute_url")

    return plain_text, absolute_url, False

In [13]:
def process_case(case):
    """Scrapes one case. Returns (rows, incomplete)."""
    name, docket, court = case["name"], case["docket_number"], case["court"]
    print(f"=== {name} ({docket}, {court}) ===")

    results, incomplete = search_docket(docket, court)
    documents = flatten_documents(results)
    print(f"  {len(documents)} fee-related documents found" + (" [SEARCH INCOMPLETE]" if incomplete else ""))
    if not documents:
        print("  WARNING: 0 documents. Check the docket_number format in cases.csv.")

    rows = []
    for doc in documents:
        plain_text, absolute_url, failed = fetch_text(doc)
        incomplete = incomplete or failed
        rows.append({
            "case_name": name, "docket_number": docket, "court": court,
            "document_id": doc.get("id"), "description": doc.get("description", ""),
            "is_available": doc.get("is_available", False),
            "plain_text": plain_text, "absolute_url": absolute_url,
        })

    with_text = sum(1 for r in rows if r["plain_text"])
    print(f"  {with_text} of {len(rows)} had free text\n")
    return rows, incomplete

## Run the Scrape
Runs only when `RUN_SCRAPE = True`. Skips cases that are already fully scraped and saves after every case, so an interrupted run loses nothing. Incomplete cases restart on the next run, and cached documents are not re-fetched.

In [14]:
if not RUN_SCRAPE:
    assert os.path.exists(RAW_PATH), f"{RAW_PATH} is missing. Download it from the shared Google Drive folder, or set RUN_SCRAPE = True."
    print("RUN_SCRAPE is False. Using the existing raw file.")
else:
    all_results = load_json(RAW_PATH) if os.path.exists(RAW_PATH) else []
    done = {(r["docket_number"], r["court"]) for r in all_results if r.get("_case_complete")}

    for case in CASES:
        key = (case["docket_number"], case["court"])
        if key in done:
            print(f"Skipping {case['name']}, already scraped")
            continue

        all_results = [r for r in all_results if (r["docket_number"], r["court"]) != key]
        rows, incomplete = process_case(case)
        for r in rows:
            r["_case_complete"] = not incomplete
        all_results.extend(rows)
        save_json(all_results, RAW_PATH)

    print(f"Done. {len(all_results)} documents total, {request_count} requests used.")

RUN_SCRAPE is False. Using the existing raw file.


## Load and Keep Documents With Text
Everything below this point works from the saved raw file and makes no API calls.

In [15]:
all_results = load_json(RAW_PATH)
with_text = [r for r in all_results if r["plain_text"]]
save_json(with_text, WITH_TEXT_PATH)

print(f"{len(with_text)} documents with text, out of {len(all_results)} total")

9936 documents with text, out of 29435 total


## Violation Categories
Keywords for tagging what examiners flag, mapped to the U.S. Trustee Appendix B guidelines (2013). The comment on each line is the guideline section. `ENTRY_LEVEL` marks categories that can be detected from a single narrative. The rest (rates, budgets, transitory timekeepers, seniority, expenses) need firm-level information.

Appendix B covers attorneys in large chapter 11 cases filed on or after October 1, 2013. Local rules and case law add to it, so treat these categories as a foundation, not a complete list. The leftover review and coverage table in Part 1 show what may be missing.

In [16]:
CATEGORIES = {
    "vague": ["vague", "insufficient detail", "lacks specificity", "lack of detail", "attention to", "review file"],    # B.2.i
    "block_billing": ["block bill", "lumping", "lumped"],                                                            # B.2.h
    "time_increments": ["tenth of an hour", "one-tenth", "half-hour increments", "whole-hour increments"],           # C.9.c
    "multiple_attendees": ["multiple attendees", "multiple professionals", "more than one professional"],            # B.2.c, C.9.f
    "overhead_clerical": ["clerical", "administrative task", "secretarial", "word processing", "summer associate"],   # B.2.j
    "billing_activities": ["time records", "preparing invoices", "invoices", "invoicing", "redact", "billing review"],  # B.2.f
    "fee_defense": ["defending the fee", "fee dispute", "explaining the fees"],                                      # B.2.g
    "nonworking_travel": ["non-working travel", "nonworking travel", "travel time"],                                 # B.2.k
    "transitory_timekeepers": ["transitory", "transient", "fewer than 15 hours"],                                    # B.2.e
    "seniority_mismatch": ["seniority", "higher billing rate", "less experienced", "lower billing rate"],            # B.2.c
    "duplication": ["duplicat", "shadowing", "up to speed", "overlap", "staffing"],                                  # B.2.c, F.5
    "rates": ["rate increase", "blended rate", "hourly rates", "customary compensation"],                            # B.2.b, B.2.d
    "internal_conferences": ["conference entries", "intra-office", "inter-office", "internal conference", "internal meeting"],  # Not a separate Appendix B item; recurs in examiner reports
    "budget_staffing_plan": ["budget", "staffing plan"],                                                             # B.2.m
    "expenses": ["first class", "first-class", "airfare", "car service", "mark-up", "markup", "unsupported expense"], # B.3
}
ENTRY_LEVEL = {"vague", "block_billing", "time_increments", "multiple_attendees", "overhead_clerical",
               "billing_activities", "fee_defense", "nonworking_travel", "internal_conferences"}

# Part 1: Fee Examiner Reports

### Filter to Real Reports
The search returns many related filings that aren't reports, such as certificates of service, notices, and reports from investigation examiners. This keeps only fee examiner and fee committee reports. The noise lists are standard filing types.

In [17]:
NOISE_PREFIXES = [
    "affidavit", "certificate", "certification", "application to employ",
    "application for pro hac vice", "declaration", "order authorizing",
    "notice of appearance", "monthly fee statement",
]
NOISE_ATTACHMENTS = ["certificate of service", "proposed order", "notice"]


def is_real_report(description):
    """True if the description looks like a fee examiner or fee committee report."""
    parent, _, attachment = (description or "").lower().partition(" | ")
    parent = parent.strip()

    if "fee examiner" not in parent and "fee committee" not in parent:
        return False
    if any(parent.startswith(p) for p in NOISE_PREFIXES):
        return False
    if any(w in attachment for w in NOISE_ATTACHMENTS):
        return False

    # "Examiner" or "committee" and "report" should appear near the start, not inside something
    # like "objection to the fee examiner's report"
    anchors = [p for p in (parent.find("examiner"), parent.find("committee")) if p >= 0]
    report_pos = parent.find("report")
    return bool(anchors) and min(anchors) < 30 and 0 <= report_pos < 100

### Deduplicate Reports
Some reports are refiled as corrected versions. This keeps the corrected version, otherwise the newest.

In [18]:
def report_key(r):
    """Groups a report with its corrected or refiled versions, but not with its own attachments."""
    parent, _, attachment = r["description"].partition(" | ")

    def norm(s):
        s = s.lower().replace("corrected ", "")
        return " ".join(re.sub(r"[^a-z ]+", " ", s).split())

    return (r["case_name"], norm(parent)[:120], norm(attachment))


def dedupe_reports(reports):
    """Keeps one report per key: the corrected version if there is one, otherwise the newest."""
    best = {}
    for r in reports:
        k = report_key(r)
        keep = best.get(k)
        r_fixed = "corrected" in r["description"].lower()
        keep_fixed = keep is not None and "corrected" in keep["description"].lower()

        if keep is None or (r_fixed and not keep_fixed) or (r_fixed == keep_fixed and r["document_id"] > keep["document_id"]):
            best[k] = r
    return list(best.values())

In [19]:
reports = dedupe_reports([r for r in with_text if is_real_report(r["description"])])
save_json(reports, f"{FINAL_DIR}/examiner_reports.json.gz")

for r in reports:
    print(f"[{r['case_name']}] {r['description'][:80]}")
print(f"\n{len(reports)} unique reports")

[Seadrill Limited] Notice Fee Examiner's Report Regarding Review of the First Interim Fee Applicati
[Seadrill Limited] Notice FEE EXAMINERS REPORT REGARDING: (I) REVIEW OF THE FINAL FEE APPLICATIONS 
[BlockFi Inc.] Document re: Fee Examiners Corrected Consolidated Final Report Regarding Final F
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Final Fee Applic
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding Final Fee Applica
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding First Interim Fee
[FTX Trading Ltd.] Examiner's Report / Fee Examiner's Summary Report on Fee Review Process and Firs
[FTX Trading Ltd.] Examiner's Report /Fee Examiner's Summary Report on Fee Review Process and Seven
[FTX Trading Ltd.] Exami

### Rebuild Paragraphs
Court filings are double spaced, so every line is followed by a blank line and splitting on blank lines doesn't work. Instead, the splitter uses two conventions court filings follow:
- Paragraphs are numbered (`45.  Proper time keeping...`).
- Sections have lettered or roman numeral headings (`D.  Lumping; Failure to Bill...`).

Headings must be in Title Case to count, since lettered list items read like sentences. Headings are only detected in some reports, so they aren't saved to the CSV, but they still help with tagging: a paragraph under "Staffing" gets tagged even if it never uses the word. Tables and long lists aren't numbered, so anything too long is split at sentence breaks.

In [20]:
PARA_START = re.compile(r"^\s*\d{1,3}\.\s+\S")                          # "45.  Proper time keeping..."
HEADING = re.compile(r"^\s*(?:[A-Z]|[IVX]{1,5})\.\s+([A-Z].{2,100})$")   # "D.  Lumping; Failure to..."
TOC_LEADER = re.compile(r"\.{4,}")                                      # Table of contents dot leaders
SENTENCE_END = re.compile(r"(?<=[.!?])[”\"]?\s+(?=[A-Z(“])")
SMALL_WORDS = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "by", "with", "at", "as", "re", "from"}


def looks_like_heading(text):
    """True if the text is in Title Case like a heading. List items read like sentences."""
    words = [w for w in re.findall(r"[A-Za-z][\w'’\-]*", text) if w.lower() not in SMALL_WORDS]
    return bool(words) and sum(w[0].isupper() for w in words) / len(words) >= 0.8


def chunk_long(paragraph, max_chars=2000):
    """Splits an overly long paragraph (usually a table or list) into sentence-sized chunks."""
    if len(paragraph) <= max_chars:
        return [paragraph]
    chunks, current = [], ""
    for sentence in SENTENCE_END.split(paragraph):
        if current and len(current) + len(sentence) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += " " + sentence
    if current.strip():
        chunks.append(current.strip())
    return chunks

In [21]:
def split_report_paragraphs(text):
    """Rebuilds paragraphs from a report. Returns a list of (section heading, paragraph text)."""
    lines = [l for l in strip_boilerplate(text).split("\n") if l.strip()]
    paragraphs, current, section = [], [], None

    def flush():
        if current:
            joined = " ".join(" ".join(current).split())
            paragraphs.extend((section, chunk) for chunk in chunk_long(joined))

    for line in lines:
        heading = HEADING.match(line)
        if (heading and not TOC_LEADER.search(line) and not line.rstrip().endswith((",", ";"))
                and looks_like_heading(heading.group(1))):
            flush()
            current = []
            section = " ".join(heading.group(1).split())
        elif PARA_START.match(line):
            flush()
            current = [line]
        elif not TOC_LEADER.search(line):
            current.append(line)

    flush()
    return paragraphs

### Paragraph Bank
Keeps paragraphs that discuss a violation, tagged by category.
- `is_guidance` marks "should / must / requires" language, which usually describes what a compliant entry looks like.
- `entry_level` marks paragraphs about violations that can be detected from a single narrative.

In [22]:
MONEY = re.compile(r"\$[\d,]+(?:\.\d{2})?")
GUIDANCE = re.compile(r"\b(should|must|require[sd]?|guidelines? provide|sufficient detail)\b", re.I)


def tag_paragraphs(report):
    """Returns one row per paragraph that mentions a violation category."""
    rows = []
    for section, p in split_report_paragraphs(report["plain_text"]):
        searchable = f"{section or ''} {p}".lower()
        hits = [c for c, kws in CATEGORIES.items() if any(k in searchable for k in kws)]
        if hits and len(p) > 80:
            rows.append({
                "case_name": report["case_name"],
                "document_id": report["document_id"],
                "categories": ",".join(hits),
                "entry_level": any(c in ENTRY_LEVEL for c in hits),
                "is_guidance": bool(GUIDANCE.search(p)),
                "amounts": ",".join(MONEY.findall(p)),
                "text": p,
                "absolute_url": report.get("absolute_url"),
            })
    return rows

In [23]:
paragraphs = pd.DataFrame([row for r in reports for row in tag_paragraphs(r)])
paragraphs.to_csv(f"{FINAL_DIR}/examiner_paragraphs.csv", index=False)

print(f"{len(paragraphs)} paragraphs tagged, {paragraphs['is_guidance'].sum()} with guidance language\n")
print(paragraphs["categories"].str.split(",").explode().value_counts())

709 paragraphs tagged, 114 with guidance language

categories
duplication               291
block_billing             174
rates                     157
vague                     156
transitory_timekeepers     84
billing_activities         71
budget_staffing_plan       58
multiple_attendees         55
overhead_clerical          54
expenses                   28
nonworking_travel          26
seniority_mismatch         19
time_increments            14
internal_conferences        4
Name: count, dtype: int64


### Report Health Check
One row per report. A very low `paragraphs` count or a small `median_len` usually means a format the splitter doesn't handle. Review that report before relying on its rows.

In [24]:
report_health = pd.DataFrame([
    {
        "case_name": r["case_name"],
        "document_id": r["document_id"],
        "paragraphs": len(ps),
        "median_len": int(pd.Series([len(p) for _, p in ps]).median()) if ps else 0,
    }
    for r in reports
    for ps in [split_report_paragraphs(r["plain_text"])]
])
report_health.sort_values("median_len")

,case_name,document_id,paragraphs,median_len
40,Energy Future Holdings Corp.,380003130,95,129
42,Energy Future Holdings Corp.,380003207,10,152
13,FTX Trading Ltd.,488992062,13,249
11,FTX Trading Ltd.,442973412,24,293
12,FTX Trading Ltd.,432002204,61,302
...,...,...,...,...
49,Energy Future Holdings Corp.,106744677,1,1580
45,Energy Future Holdings Corp.,380086112,4,1689
46,Energy Future Holdings Corp.,34400272,5,1855
39,Energy Future Holdings Corp.,380083207,2,2851


### Untagged Flagged Paragraphs
Paragraphs that use flagging language ("reduce," "object," "non-compensable") but match no category. Most are generic, such as "concerns were resolved through a negotiated adjustment." This is where new violation types show up. After adding cases, read the per-case sample below. If the same specific issue appears across cases, add a category for it.

In [25]:
FLAG_WORDS = re.compile(r"\b(reduc\w*|disallow\w*|object\w*|non-?compensable|not compensable|excessive|deduct\w*)\b", re.I)


def is_tagged(section, paragraph):
    """True if the paragraph or its section heading matches any violation category."""
    searchable = f"{section or ''} {paragraph}".lower()
    return any(k in searchable for kws in CATEGORIES.values() for k in kws)

In [26]:
leftover = pd.DataFrame([
    {"case_name": r["case_name"], "section": section, "text": p}
    for r in reports
    for section, p in split_report_paragraphs(r["plain_text"])
    if FLAG_WORDS.search(p) and not is_tagged(section, p) and len(p) > 80
])
print(f"{len(leftover)} flagged paragraphs with no category")
leftover.sample(min(len(leftover), 20), random_state=0) if len(leftover) else leftover

571 flagged paragraphs with no category


,case_name,section,text
127,Celsius Network LLC,NaN,16. Retention Activities. Each Retained Profes...
337,Boy Scouts of America,NaN,15. After accounting for the agreed fee reduct...
312,Boy Scouts of America,NaN,12. Rucki Fee Review identified certain confer...
532,Voyager Digital Holdings Inc.,NaN,Independent Fee Examiner Lori Lapin Jones PLLC...
361,Boy Scouts of America,NaN,14. In addition to those other matters referen...
477,Boy Scouts of America,NaN,11. Rucki Fee Review identified certain confer...
134,Celsius Network LLC,NaN,"64. Through Katten, PWP either provided supple..."
170,Energy Future Holdings Corp.,NaN,A complete list of such information may be obt...
262,Boy Scouts of America,NaN,14. Rucki Fee Review requested a reduction on ...
210,Sears Holdings Corp.,NaN,"2. On April 22, 2019, the Court entered its Or..."


In [27]:
# Five untagged paragraphs per case, for manual review
(leftover.sample(frac=1, random_state=0).groupby("case_name").head(5)
    .sort_values("case_name")[["case_name", "text"]] if len(leftover) else leftover)

,case_name,text
64,BlockFi Inc.,"45. During the Fee Period, Deloitte Tax filed ..."
66,BlockFi Inc.,52. After her review of the Kroll Fee Applicat...
45,BlockFi Inc.,8. Under paragraphs 5 and 10 of the Fee Examin...
46,BlockFi Inc.,16. The Fee Examiner limited her review to the...
71,BlockFi Inc.,29. HB and the Fee Examiner engaged in a produ...
337,Boy Scouts of America,15. After accounting for the agreed fee reduct...
312,Boy Scouts of America,12. Rucki Fee Review identified certain confer...
361,Boy Scouts of America,14. In addition to those other matters referen...
477,Boy Scouts of America,11. Rucki Fee Review identified certain confer...
262,Boy Scouts of America,14. Rucki Fee Review requested a reduction on ...


### Category Coverage
Which categories appear in each case's reports. This measures coverage of the existing categories but can't surface new violation types; the untagged review above does that. A category that never appears either has the wrong keywords or is rarely raised.

In [28]:
coverage = pd.DataFrame([
    {"case_name": r["case_name"], "category": c}
    for r in reports
    for section, p in split_report_paragraphs(r["plain_text"])
    for c, kws in CATEGORIES.items()
    if any(k in f"{section or ''} {p}".lower() for k in kws)
]).drop_duplicates()

print(f"Categories never seen: {sorted(set(CATEGORIES) - set(coverage['category']))}")
pd.crosstab(coverage["case_name"], coverage["category"])

Categories never seen: ['fee_defense']


category,billing_activities,block_billing,budget_staffing_plan,duplication,expenses,internal_conferences,multiple_attendees,nonworking_travel,overhead_clerical,rates,seniority_mismatch,time_increments,transitory_timekeepers,vague
case_name,,,,,,,,,,,,,,
BlockFi Inc.,1,1,1,1,1,0,0,0,1,1,0,1,1,1
Boy Scouts of America,1,1,1,1,0,1,1,1,1,1,1,0,1,1
Celsius Network LLC,1,1,1,1,1,0,1,1,1,1,1,1,1,1
Energy Future Holdings Corp.,0,0,0,1,1,0,1,0,1,1,0,0,1,1
FTX Trading Ltd.,1,1,1,1,1,1,1,1,1,1,1,1,1,1
PG&E Corp.,1,0,0,1,0,0,0,1,0,0,0,0,0,0
Purdue Pharma L.P.,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Seadrill Limited,1,1,1,1,1,0,1,1,1,1,1,1,0,1
Sears Holdings Corp.,1,0,1,0,0,0,0,0,0,0,0,0,0,0


# Part 2: Time Entries

### Date Formats
Entries start with a date. Supported formats: `1/17/23`, `01-17-2023`, `2023-01-17`, and `Jan. 17, 2023`. The same pattern finds the entry section and splits it into entries.

In [29]:
MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?"
DATE_REGEX = (
    r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}"   # 1/17/23, 01-17-2023
    r"|\d{4}-\d{2}-\d{2}"               # 2023-01-17
    rf"|{MONTH}\s+\d{{1,2}},?\s+\d{{4}})"  # Jan. 17, 2023
)
DATE_PATTERN = re.compile(rf"^{DATE_REGEX}(?=\s|$)")
DATE_LINE = re.compile(rf"^\s*{DATE_REGEX}(?=\s|$)", re.M)

### Find the Time-Entry Section
Fee statements put itemized entries in an exhibit. The finder first looks for a marker like "time detail" followed by date lines, which skips table of contents mentions. If no marker matches, it falls back to the first long run of date lines. Either way, the section ends before the expense detail. `found_by` records which method worked.

In [30]:
TIME_DETAIL_MARKERS = ["time detail", "detailed time records", "time log", "daily time log"]
END_MARKERS = ["expense detail", "summary of expenses", "disbursement", "expenses incurred"]
MIN_DATE_LINES = 25  # Fallback needs a real run of entries, not a small summary table


def cut_at_expenses(section):
    """Trims a section right before the expense detail starts."""
    ends = [i for i in (section.lower().find(e) for e in END_MARKERS) if i > 0]
    return section[:min(ends)] if ends else section

In [31]:
def find_time_entry_section(text):
    """Returns (section, found_by) for the itemized time entries, or (None, None) if there are none."""
    lower = text.lower()
    for marker in TIME_DETAIL_MARKERS:
        for m in re.finditer(re.escape(marker), lower):
            if DATE_LINE.search(text, m.end(), m.end() + 3000):
                return cut_at_expenses(text[m.start():]), "marker"

    date_lines = list(DATE_LINE.finditer(text))
    if len(date_lines) >= MIN_DATE_LINES:
        return cut_at_expenses(text[date_lines[0].start():]), "date_lines"

    return None, None

### Who Billed It
Only attorney entries are kept. Financial advisors and claims agents also bill, but their entries read differently. Instead of listing firm names, the classifier looks at how the filing describes the applicant. PACER descriptions often say "filed by Financial Advisor...," and fee statements describe the applicant as "financial advisor to the Official Committee" and so on.

Almost every filing has a footnote mentioning the "debtors' claims and noticing agent," so a role only counts when it's followed by "to the debtors" or "for the committee," which is how applicants describe themselves.

In [32]:
def extract_firm(description):
    """Best guess at the billing firm from the filing description. Returns None if unclear."""
    d = " ".join((description or "").split())
    m = re.search(
        r"(?:statement|application) of ([A-Z][\w&.,' ]+?(?:LLP|LLC|L\.L\.P\.|P\.C\.|P\.A\.|Inc\.?)(?![a-z]))",
        d, re.I,
    )
    if m:
        return m.group(1).strip(" ,")
    m = re.search(r"[Ff]iled by (?:Attorney |Other Professional )?([^.(|]+)", d)
    return m.group(1).strip() if m else None

In [33]:
ROLE_IN_DESCRIPTION = re.compile(
    r"filed by (financial advisor|claims and noticing agent|claims agent|accountant|consultant|auditor)", re.I)
ROLE_IN_TEXT = re.compile(
    r"(financial advis[eo]rs?|investment bank(?:ers?)?|accountants?|consultants?|"
    r"claims (?:and |& )?(?:noticing |solicitation )?agent|noticing agent)"
    r"\s+(?:to|for)\s+(?:the\s+)?(?:debtor|official|ad hoc|committee|chapter|fee examiner|examiner|unsecured)", re.I)


def classify_professional(description, text):
    """Guesses who billed from how the filing describes the applicant.
    Returns 'attorney', 'advisor', or 'claims_agent'. Defaults to 'attorney' if nothing matches."""
    m = ROLE_IN_DESCRIPTION.search(description or "") or ROLE_IN_TEXT.search((text or "")[:5000])
    if not m:
        return "attorney"
    return "claims_agent" if "agent" in m.group(1).lower() else "advisor"

### Pull Out the Sections

In [34]:
report_ids = {r["document_id"] for r in reports}  # Reports have dated summary tables, so skip them

sections = []
for r in with_text:
    if r["document_id"] in report_ids:
        continue
    text = strip_boilerplate(r["plain_text"])
    section, found_by = find_time_entry_section(text)
    if section:
        sections.append({
            "case_name": r["case_name"],
            "firm": extract_firm(r["description"]),
            "role": classify_professional(r["description"], text),
            "found_by": found_by,
            "document_id": r["document_id"],
            "description": r["description"],
            "raw_entry_section": section,
            "absolute_url": r.get("absolute_url"),
        })

save_json(sections, SECTIONS_PATH)
print(f"{len(sections)} documents have a time-entry section, out of {len(with_text)} with text\n")
print(pd.DataFrame(sections)[["role", "found_by"]].value_counts())

2149 documents have a time-entry section, out of 9936 with text

role          found_by  
attorney      date_lines    1163
              marker         697
advisor       date_lines     151
              marker         136
claims_agent  date_lines       2
Name: count, dtype: int64


### Parsing Patterns
Hours can look like `.4`, `0.40`, or `5.9`. Task codes are UTBMS codes like `L320` or `A104`, the standard legal billing codes. Table header rows that leak into narratives are stripped.

Many narratives list time per task in parentheses, like "draft motion (1.2); call with client (.3)." These splits are never read as the entry's hours. Hours come from the hours column. If the column can't be found, the split total is used instead.

In [35]:
HOURS_PATTERN = re.compile(r"(?<![\d.,$(])(\d{0,2}\.\d{1,2})(?![\d,)])")  # Skips "(1.2)" task splits
# Task splits like "(1.2)", "( .2)", "(.40 - partial)", or "(1);". Whole numbers only count when followed by
# punctuation, so references like "30(b)(6) notice" aren't read as time
SPLIT_PATTERN = re.compile(r"\(\s*(\d{0,2}\.\d{1,2})\s*(?:-[^)]*)?\)|\((\d{1,2})\)(?=\s*(?:[;,.]|$))")
AMOUNT_PATTERN = re.compile(r"(?<![\w(])\$?\s*[\d,]*\.\d{1,2}(?![\w)])")  # Leaves "(0.8)" time splits alone
TASK_CODE = re.compile(r"\b([ABEL]\d{3})\b")

NOT_NAMES = {"date", "mon", "tue", "wed", "thu", "fri", "sat", "sun",
             "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
ADMIN_WORDS = {"invoice", "total", "matter", "page", "client", "balance", "remittance", "desc"}

# When an entry runs across a page break, the next page's table header can land in the middle of it
# ("...and Prime Date Tkpr Task Hours Value Trust regarding same"). Every billing table has a header
# row like this, so any run of 3+ header words is stripped, along with invoice and page numbers
HEADER_WORDS = r"(?:Date|Atty|Attorney|Task|Tkpr|Timekeeper|Professional|Person|Name|Description|Services|Rendered|Narrative|Hours|Hrs|Rate|Amount|Value|Of)"
TABLE_HEADER = re.compile(rf"(?:\b{HEADER_WORDS}\b[\s:.]*){{3,}}", re.I)
INVOICE_BITS = re.compile(r"\bInvoice\s*(?:No\.?|#|Number)\s*:?\s*[\d-]+|\bPage:?\s*\d+(?:\s+of\s+\d+)?", re.I)

### Parsing Helpers
Small checks used by the entry parser.

In [36]:
def looks_like_a_name(text):
    """True if a column looks like a timekeeper: a name, initials, or "Last, First"."""
    if not text or len(text) > 40:
        return False
    lower = text.lower()
    if lower.strip(" .") in NOT_NAMES or any(w in lower for w in ADMIN_WORDS):
        return False
    if any(c.isdigit() for c in text):
        return False
    return bool(re.fullmatch(r"[A-Za-z.,'\-\s]+", text))


def split_total(text):
    """Sums the per-task time splits in a narrative. Returns None if there are fewer than two."""
    values = [float(dec or whole) for dec, whole in SPLIT_PATTERN.findall(text)]
    return round(sum(values), 2) if len(values) >= 2 else None


def clean_narrative(text):
    """Strips dollar amounts, bracketed notes, table headers, and leading ID numbers from a narrative."""
    text = AMOUNT_PATTERN.sub(" ", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = TABLE_HEADER.sub(" ", text)
    text = INVOICE_BITS.sub(" ", text)
    text = " ".join(text.split()).strip()
    return re.sub(r"^(?:\d{3,10}\s+)+", "", text)  # Can be more than one, like a task code then an entry ID

In [37]:
def split_into_entry_blocks(text):
    """Splits a section into one block per entry. A new entry starts at each line that begins with a date."""
    blocks, current = [], []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if DATE_PATTERN.match(line):
            if current:
                blocks.append("\n".join(current))
            current = [line]
        elif current:
            current.append(line)
    if current:
        blocks.append("\n".join(current))
    return blocks

### The Entry Parser
Splits on wide column gaps instead of matching name patterns, so capitalization doesn't matter. Low-confidence entries keep their `raw_block` instead of a guessed narrative, so nothing is silently wrong.

In [38]:
def parse_entry_block(block):
    """Turns one raw entry block into date, timekeeper, hours, task codes, and narrative.
    hours_check compares the hours to the per-task splits: 'match', 'mismatch', or 'no_splits'."""
    date_match = DATE_PATTERN.match(block)
    date = date_match.group(1) if date_match else None
    remainder = block[date_match.end():] if date_match else block

    task_codes = TASK_CODE.findall(remainder.split("\n")[0])
    remainder = TASK_CODE.sub(" ", remainder)

    columns = [c.strip() for c in re.split(r"\s{2,}", remainder) if c.strip()]
    timekeeper = columns[0] if columns else None
    rest = " ".join(columns[1:])

    numbers = [float(n) for n in HOURS_PATTERN.findall(rest.split("\n")[0])]
    hours = next((n for n in numbers if 0 < n <= 24), None)

    splits = split_total(rest)
    if hours is None and splits and splits <= 24:
        hours = splits
    if hours is None:
        hours_check = None
    elif splits is None:
        hours_check = "no_splits"
    else:
        hours_check = "match" if abs(hours - splits) <= 0.05 else "mismatch"

    narrative = clean_narrative(rest)
    name_ok = looks_like_a_name(timekeeper)
    high_confidence = bool(date and name_ok and hours is not None and len(narrative) > 5)

    return {
        "date": date,
        "timekeeper": timekeeper if name_ok else None,
        "hours": hours,
        "task_codes": ",".join(task_codes),
        "split_total": splits,
        "hours_check": hours_check,
        "narrative": narrative if high_confidence else None,
        "high_confidence": high_confidence,
        "raw_block": block,
    }

In [39]:
all_entries = []
for doc in sections:
    for block in split_into_entry_blocks(doc["raw_entry_section"]):
        parsed = parse_entry_block(block)
        parsed.update({
            "case_name": doc["case_name"],
            "firm": doc["firm"],
            "role": doc["role"],
            "found_by": doc["found_by"],
            "document_id": doc["document_id"],
            "description": doc["description"],
            "absolute_url": doc["absolute_url"],
        })
        all_entries.append(parsed)

entries = pd.DataFrame(all_entries)
print(f"{len(entries)} entries parsed, {entries['high_confidence'].mean():.0%} high confidence")

559626 entries parsed, 72% high confidence


### Double-Check Who Billed It
The role check above only works when a filing states the applicant's role, and many don't. For example, a New Jersey description may only say "filed by [attorney] on behalf of M3 Partners." This second check looks at what the entries discuss. Attorney entries center on legal work (motions, hearings, drafting), advisor entries on financial work (analysis, models, decks), and claims agent entries on creditor inquiries and mailings.

Scoring is per document, since a single entry is too short to judge. Validated against filings with a named applicant, it caught about 92% of advisor entries and 95% of claims agent entries while keeping about 93% of attorney entries. Most attorney entries it drops come from fee examiner counsel reviewing other firms' bills, which isn't typical billing.

In [40]:
LEGAL_WORDS = re.compile(
    r"\b(?:motions?|briefs?|pleadings?|draft\w*|revis\w*|hearing|court|objections?|stipulation|order|research|"
    r"depositions?|discovery|privilege|memo\w*|complaint|docket|counsel|settlement|declaration|agreement|"
    r"statute|judge|chambers|filing|file)\b", re.I)
ADVISOR_WORDS = re.compile(
    r"\b(?:analys[ie]s|analyz\w*|model\w*|deck|presentation|waterfall|forecast\w*|cash|budget|reconcil\w*|"
    r"database|data|slides?|valuation|spreadsheet|dashboard|metrics|MOR|balance sheet|financial)\b", re.I)
AGENT_WORDS = re.compile(
    r"\b(?:creditor inquir\w*|inquiries|call center|noticing|mailing|ballots?|solicitation|claims register|"
    r"claim forms?|service list|tabulat\w*)\b", re.I)

MIN_ENTRIES_TO_JUDGE = 5   # Documents with fewer clean entries are too small to score
AGENT_SHARE = 0.25         # A document is a claims agent's if a quarter of its entries are claims agent work

In [41]:
def guess_role_from_narratives(entries):
    """Scores each document by what its entries discuss. Returns a Series mapping
    document_id to 'attorney', 'advisor', 'claims_agent', or 'too_few'."""
    hc = entries[entries["high_confidence"]]
    shares = pd.DataFrame({
        "document_id": hc["document_id"],
        "legal": hc["narrative"].str.contains(LEGAL_WORDS),
        "advisor": hc["narrative"].str.contains(ADVISOR_WORDS),
        "agent": hc["narrative"].str.contains(AGENT_WORDS),
    }).groupby("document_id").agg(
        n=("legal", "size"), legal=("legal", "mean"), advisor=("advisor", "mean"), agent=("agent", "mean"))

    def pick(r):
        if r["n"] < MIN_ENTRIES_TO_JUDGE:
            return "too_few"
        if r["agent"] >= AGENT_SHARE:
            return "claims_agent"
        return "advisor" if r["advisor"] > r["legal"] else "attorney"

    return shares.apply(pick, axis=1)

In [42]:
entries["vocab_role"] = entries["document_id"].map(guess_role_from_narratives(entries)).fillna("too_few")

# Use the filing's stated role first, then fall back to the vocabulary check
entries["final_role"] = np.where(
    entries["role"] != "attorney", entries["role"],
    np.where(entries["vocab_role"].isin(["advisor", "claims_agent"]), entries["vocab_role"], "attorney"),
)
print(pd.crosstab(entries["role"], entries["vocab_role"]))
print()
print(entries["final_role"].value_counts())

vocab_role    advisor  attorney  claims_agent  too_few
role                                                  
advisor         22826      2968             0     1548
attorney        95425    348918         13955    73592
claims_agent        0         0             0      394

final_role
attorney        422510
advisor         122767
claims_agent     14349
Name: count, dtype: int64


### Tag, Deduplicate, and Balance
- **Tag:** only attorney-billed entries are kept, based on `final_role`. Expense lines that slip into the time section are dropped, along with entries that start mid-sentence. Some layouts put the date partway through an entry, so those splits land in the wrong place. An entry only counts as an expense if it's short and expense-like, so a real entry like "research on Westlaw re: preference claims" is kept. Entries whose per-task splits don't add up to the hours are also dropped, since the true total is ambiguous.
- **Deduplicate:** interim applications re-attach monthly statements, so the same entry can appear more than once.
- **Balance:** each case is capped at `MAX_ENTRIES_PER_CASE` so one large case doesn't dominate. Cases reviewed and found unusable go in `EXCLUDE_FROM_BALANCED` and stay in `time_entries_all`. If a case keeps producing unusable entries in the spot check, exclude it rather than writing new rules for its layout.

In [43]:
EXPENSE_WORDS = re.compile(r"charges?|photocopy|copies|postage|courier|filing fee|lexis|westlaw|pacer|meals?|airfare|hotel|"
                           r"reproduction|taxi|transportation|mileage|parking|lodging", re.I)
MAX_EXPENSE_WORDS = 8


def is_expense_line(narrative):
    """True for short lines like 'Telecommunication Charges' that are expenses, not time."""
    if not isinstance(narrative, str):
        return False
    return len(narrative.split()) <= MAX_EXPENSE_WORDS and bool(EXPENSE_WORDS.search(narrative))


def tag_entry_type(df):
    """Adds an entry_type column: 'attorney' for attorney-billed time, 'exclude' for everything else.
    Entries that start mid-sentence or whose splits don't add up to the hours are excluded."""
    is_expense = df["narrative"].apply(is_expense_line)
    starts_mid = df["narrative"].str.match(r"^[a-z(]", na=False)
    hours_mismatch = df["hours_check"] == "mismatch"
    keep = (df["final_role"] == "attorney") & ~is_expense & ~starts_mid & ~hours_mismatch
    df["entry_type"] = np.where(keep, "attorney", "exclude")
    return df

In [44]:
entries = tag_entry_type(entries)
entries = entries.drop_duplicates(["case_name", "date", "timekeeper", "hours", "raw_block"])
entries.to_csv(f"{FINAL_DIR}/time_entries_all.csv.gz", index=False)  # Gzipped to stay under GitHub's file limit

clean = entries[entries["high_confidence"] & (entries["entry_type"] == "attorney")]
# Cases reviewed and left out of the balanced set. They remain in time_entries_all.
# Energy Future Holdings: most parsed rows are expense lines from a layout the parser misreads.
# Boy Scouts of America: one parsed entry; the case contributes examiner reports instead.
EXCLUDE_FROM_BALANCED = ["Energy Future Holdings Corp.", "Boy Scouts of America"]

balanced = (clean[~clean["case_name"].isin(EXCLUDE_FROM_BALANCED)]
            .sample(frac=1, random_state=42)
            .groupby("case_name").head(MAX_ENTRIES_PER_CASE))
print(f"{len(entries)} unique entries, {len(clean)} clean attorney entries, {len(balanced)} in the balanced set")

493962 unique entries, 232431 clean attorney entries, 22270 in the balanced set


### Export in Synthetic Dataset Format
The balanced set is saved with the same column names and conventions as the synthetic dataset (`synthetic_time_entries_v2.xlsx`), so real and synthetic entries can be compared side by side.
- Shared fields use the synthetic names and formats: `Time ID`, `Timekeeper`, `Date` (MM/DD/YYYY), `Hours`, and `Narrative`. Real entries get `R-` IDs so they never collide with synthetic `T-` IDs.
- Fields public filings don't include (`Status`, `TK ID`, `CMID`, `Client`, `Matter`) are left out rather than invented.
- `SOURCE:` columns record where each entry came from. Like the synthetic `KEY:` columns, drop them before feeding a model.

In [45]:
def to_synthetic_format(df):
    """Formats entries to match the synthetic dataset's column names and conventions."""
    df = df.assign(_date=pd.to_datetime(df["date"], format="mixed", errors="coerce"))
    df = df.sort_values(["case_name", "_date"]).reset_index(drop=True)

    urls = df["absolute_url"].fillna("")
    return pd.DataFrame({
        "Time ID": [f"R-{i:06d}" for i in range(1, len(df) + 1)],
        "Timekeeper": df["timekeeper"],
        "Date": df["_date"].dt.strftime("%m/%d/%Y"),
        "Hours": df["hours"],
        "Narrative": df["narrative"],
        "SOURCE: Case": df["case_name"],
        "SOURCE: Firm": df["firm"],
        "SOURCE: Task Codes": df["task_codes"].replace("", np.nan),
        "SOURCE: URL": np.where(urls != "", "https://www.courtlistener.com" + urls, None),
    })

In [46]:
export = to_synthetic_format(balanced)
export.to_csv(f"{FINAL_DIR}/time_entries_clean_balanced.csv", index=False)

print(f"Saved {len(export)} rows. {export['Date'].isna().sum()} dates couldn't be parsed.")
export.head()

Saved 22270 rows. 0 dates couldn't be parsed.


,Time ID,Timekeeper,Date,Hours,Narrative,SOURCE: Case,SOURCE: Firm,SOURCE: Task Codes,SOURCE: URL
0,R-000001,GB,12/14/2022,0.5,DI Review and revise 327 application Retention...,BlockFi Inc.,Michael D,NaN,https://www.courtlistener.com/docket/66497849/...
1,R-000002,GSK,12/29/2022,2.0,"Review docket; review first-day motions, etc.",BlockFi Inc.,Daniel Stolz on behalf of Genova Burns LLC,B110,https://www.courtlistener.com/docket/66497849/...
2,R-000003,GSK,12/29/2022,0.1,Tel with DMS re BlockFi Committee representation.,BlockFi Inc.,Daniel Stolz on behalf of Genova Burns LLC,B110,https://www.courtlistener.com/docket/66497849/...
3,R-000004,JGB,12/29/2022,0.4,DI Correspond with M. Henry (BlockFi) in conne...,BlockFi Inc.,Michael D,NaN,https://www.courtlistener.com/docket/66497849/...
4,R-000005,SAWYER,12/29/2022,0.9,COMMITTEE CALL RE INITIAL ACTION ITEMS,BlockFi Inc.,Daniel Stolz on behalf of Brown Rudnick LLP,NaN,https://www.courtlistener.com/docket/66497849/...


# Quality Checks

### Health Check per Case
One row per case. A low `pct_high_conf` usually means a format the parser hasn't seen. Cases with 0 `with_text` have no free documents on RECAP. `excluded` counts entries dropped as non-attorney, expense, mis-split, or with hours that don't match their splits.

In [47]:
raw_df = pd.DataFrame(all_results)
health = raw_df.groupby("case_name").agg(
    docs=("document_id", "nunique"),
    with_text=("plain_text", lambda s: s.notna().sum()),
)
health = health.join([
    pd.DataFrame(sections).groupby("case_name").size().rename("entry_docs"),
    entries.groupby("case_name").agg(entries=("date", "size"), pct_high_conf=("high_confidence", "mean")),
    entries[entries["entry_type"] == "exclude"].groupby("case_name").size().rename("excluded"),
    clean.groupby("case_name").size().rename("clean_entries"),
    pd.DataFrame(reports).groupby("case_name").size().rename("reports"),
]).fillna(0)

health["pct_high_conf"] = (health["pct_high_conf"] * 100).round(0)
health.sort_values("pct_high_conf")

,docs,with_text,entry_docs,entries,pct_high_conf,excluded,clean_entries,reports
case_name,,,,,,,,
Caesars Entertainment Operating Co.,134,0,0.0,0.0,0.0,0.0,0.0,0.0
Intelsat S.A.,843,0,0.0,0.0,0.0,0.0,0.0,0.0
Seadrill Limited,128,2,0.0,0.0,0.0,0.0,0.0,2.0
"Toys ""R"" Us, Inc.",611,0,0.0,0.0,0.0,0.0,0.0,0.0
Energy Future Holdings Corp.,8763,3559,187.0,6788.0,18.0,1122.0,188.0,23.0
Celsius Network LLC,964,364,142.0,17284.0,63.0,5207.0,7377.0,6.0
PG&E Corp.,3288,1585,187.0,197827.0,65.0,50480.0,81965.0,2.0
Boy Scouts of America,3004,109,3.0,85.0,73.0,0.0,62.0,49.0
FTX Trading Ltd.,1746,1018,197.0,73566.0,76.0,29535.0,26868.0,12.0


### Spot Check
Twenty random clean entries per case. Review these after adding cases. If something looks wrong, find the general pattern before adding a fix.

In [48]:
for case, group in balanced.groupby("case_name"):
    print(f"\n### {case}")
    for _, r in group.sample(min(len(group), 20), random_state=0).iterrows():
        print(f"{r['date']} | {r['timekeeper']} | {r['hours']} | {r['task_codes']} | {r['narrative'][:150]}")


### BlockFi Inc.
10/10/23 | DMS | 0.2 | B150 | Text to creditors committee regarding expense reimbursement.
03/09/23 | DWOSKIN | 0.4 |  | CALL WITH ELEMENTS RE ASSETS SUBJECT TO SEIZURE
01/28/23 | Stephanie A. Pisko | 1.5 |  | High-level review of Bankruptcy docket in preparation of preparing research and memorandum on matter.
5/21/23 | GSK | 0.1 | B110 | Review various Court notices.
2/17/23 | DMS | 0.4 | B110 | Emails with Stark and Aulet regarding preparation for February 21st.
5/18/23 | LD | 2.0 | B110 | Preparation of adversary complaint, exhibits, motion to seal, in anticipation of filing.
09/26/23 | SILVERBERG | 4.9 |  | CONFIRMATION HEARING PREPARATIONS (.4); ATTEND CONFIRMATION HEARING (VIRTUAL AND IN PERSON) (4.5)
02/14/23 | JOYNSON | 3.1 |  | ATTEND AND DRAFT INTERVIEW SUMMARY OF INTERVIEW OF BLOCKFI EXECUTIVES
08/29/23 | Jessica J. | 0.6 |  | Preparing detailed comments to fee Macarone examiner's inquiries for 10 inquiries regarding duplicate entries.
1/17/23 | DMS | 0.2 |